# Driver Churn Prediction — Model Training

This notebook contains the complete model-training workflow used by the Streamlit dashboard.

Run the cells from top to bottom. It creates `churn_model.pkl` and `model_metrics.pkl`.


In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pathlib import Path

BASE_DIR = Path.cwd()

DATA_PATH = BASE_DIR / "driver_data.csv"
MODEL_PATH = BASE_DIR / "churn_model.pkl"
METRICS_PATH = BASE_DIR / "model_metrics.pkl"

print("Working directory:", BASE_DIR)

# Keep this date identical to the dashboard's feature-engineering reference date.
CURRENT_DATE = pd.Timestamp("2025-12-24")

FINAL_FEATURES = [
    "inactivity_days",
    "activeduration_days",
    "trips_per_day",
    "earnings_per_trip",
    "online_efficiency",
    "recent_activity_flag",
    "earnings_to_trip_ratio",
    "incentive_engagement",
    "engagement_x_trips",
    "utilisation_x_hours",
    "earnings_x_engagement",
    "cancellationrate",
    "engagementscore",
    "incentiveparticipation",
    "missing_ratings",
    "missing_averagedailyearnings_ghs",
    "missing_utilisation",
    "city",
    "cancellation_bin",
    "active_hour_pattern",
    "high_value_driver",
    "at_risk_driver",
    "new_driver_flag",
]

NUMERIC_FEATURES = [
    "inactivity_days",
    "activeduration_days",
    "trips_per_day",
    "earnings_per_trip",
    "online_efficiency",
    "recent_activity_flag",
    "earnings_to_trip_ratio",
    "incentive_engagement",
    "engagement_x_trips",
    "utilisation_x_hours",
    "earnings_x_engagement",
    "cancellationrate",
    "engagementscore",
    "incentiveparticipation",
    "missing_ratings",
    "missing_averagedailyearnings_ghs",
    "missing_utilisation",
    "high_value_driver",
    "at_risk_driver",
    "new_driver_flag",
]

CATEGORICAL_FEATURES = [
    "city",
    "cancellation_bin",
    "active_hour_pattern",
]


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Apply the same feature engineering used by the dashboard."""
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(" ", "_").str.strip()


    df["signupdate"] = pd.to_datetime(df["signupdate"], errors="coerce")
    df["lastactivedate"] = pd.to_datetime(df["lastactivedate"], errors="coerce")
    df["lastactivedate"] = df["lastactivedate"].fillna(CURRENT_DATE)

    df["activeduration_days"] = (
        df["lastactivedate"] - df["signupdate"]
    ).dt.days.clip(lower=0)

    df["inactivity_days"] = (
        CURRENT_DATE - df["lastactivedate"]
    ).dt.days.clip(lower=0)

    missing_cols = [
        "ratings",
        "averagedailyearnings_ghs",
        "utilisation",
        "tripsperweek",
        "hoursonlineperday",
        "earningsperweek",
        "cancellationrate",
        "engagementscore",
    ]

    for col in missing_cols:
        if col in df.columns:
            df[f"missing_{col}"] = df[col].isna().astype(int)

    numeric_cols = [
        "dayssincelasttrip",
        "totaltrips",
        "tripsperweek",
        "hoursonlineperday",
        "utilisation",
        "averagedailyearnings_ghs",
        "earningsperweek",
        "ratings",
        "cancellationrate",
        "incentiveparticipation",
        "engagementscore",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].fillna(df[col].median())

    df["trips_per_day"] = (
        df["totaltrips"] /
        df["activeduration_days"].replace(0, np.nan)
    )

    df["daily_trips"] = df["tripsperweek"] / 7

    df["earnings_per_trip"] = (
        df["averagedailyearnings_ghs"] /
        df["daily_trips"].replace(0, np.nan)
    )

    df["online_efficiency"] = (
        df["utilisation"] * df["hoursonlineperday"]
    )

    df["recent_activity_flag"] = (
        df["inactivity_days"] > 30
    ).astype(int)

    df["cancellation_bin"] = pd.cut(
        df["cancellationrate"],
        bins=[0, 0.2, 0.35, 1],
        labels=["Low", "Medium", "High"],
        include_lowest=True,
    )

    df["earnings_to_trip_ratio"] = (
        df["earningsperweek"] /
        df["tripsperweek"].replace(0, np.nan)
    )

    df["incentive_engagement"] = (
        df["incentiveparticipation"] *
        df["engagementscore"]
    )

    df["active_hour_pattern"] = pd.cut(
        df["hoursonlineperday"],
        bins=[0, 6, 12, 24],
        labels=["Part-time", "Full-time", "Over-time"],
        include_lowest=True,
    )

    df["engagement_x_trips"] = (
        df["engagementscore"] * df["tripsperweek"]
    )

    df["utilisation_x_hours"] = (
        df["utilisation"] * df["hoursonlineperday"]
    )

    df["earnings_x_engagement"] = (
        df["averagedailyearnings_ghs"] *
        df["engagementscore"]
    )

    df["new_driver_flag"] = (
        df["activeduration_days"] < 90
    ).astype(int)

    # These two flags are model inputs. Calculate them using the training
    # population's medians rather than leaving them as constants.
    totaltrips_median = df["totaltrips"].median()
    earnings_median = df["earningsperweek"].median()
    cancellation_median = df["cancellationrate"].median()
    inactivity_median = df["inactivity_days"].median()
    engagement_median = df["engagementscore"].median()

    df["high_value_driver"] = (
        (df["totaltrips"] > totaltrips_median)
        & (df["earningsperweek"] > earnings_median)
        & (df["cancellationrate"] < cancellation_median)
    ).astype(int)

    df["at_risk_driver"] = (
        (df["inactivity_days"] > inactivity_median)
        & (df["engagementscore"] < engagement_median)
        & (df["cancellationrate"] > cancellation_median)
    ).astype(int)

    return df


def build_pipeline() -> Pipeline:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )


def main():
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Could not find {DATA_PATH.name}. Put the CSV beside this script."
        )

    print("Loading data...")
    raw_df = pd.read_csv(DATA_PATH)

    if "churn" not in raw_df.columns:
        raise ValueError("The dataset must contain a 'churn' target column.")

    df = engineer_features(raw_df)

    missing_features = [f for f in FINAL_FEATURES if f not in df.columns]
    if missing_features:
        raise ValueError(
            "The following model features are missing after feature engineering:\n"
            + "\n".join(missing_features)
        )

    X = df[FINAL_FEATURES].copy()
    y = pd.to_numeric(df["churn"], errors="coerce")

    valid = y.notna()
    X = X.loc[valid]
    y = y.loc[valid].astype(int)

    print(f"Rows: {len(X):,}")
    print(f"Churn rate: {y.mean():.2%}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42,
    )

    pipeline = build_pipeline()

    print("Training Logistic Regression...")
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, y_prob)),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "test_size": int(len(y_test)),
        "train_size": int(len(y_train)),
        "feature_count": len(FINAL_FEATURES),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
    )
    metrics["cv_roc_auc_mean"] = float(cv_scores.mean())
    metrics["cv_roc_auc_std"] = float(cv_scores.std())

    with open(MODEL_PATH, "wb") as f:
        pickle.dump(
            {
                "pipeline": pipeline,
                "features": FINAL_FEATURES,
                "numeric_features": NUMERIC_FEATURES,
                "categorical_features": CATEGORICAL_FEATURES,
                "current_date": str(CURRENT_DATE.date()),
            },
            f,
        )

    with open(METRICS_PATH, "wb") as f:
        pickle.dump(metrics, f)

    print("\nModel saved:")
    print(MODEL_PATH)
    print("\nModel metrics:")
    for key, value in metrics.items():
        print(f"{key}: {value}")


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'sklearn'